Part 3 - WATER SYSTEMS

Scrape the search results page from https://dww.des.sc.gov/DWW/ Links to an external site.which includes submitting the form - and include the link. Save as a CSV. Since it's more than just the text from the table, this requires actually using BeautifulSoup :(

Tips:

We talked about form submission being for Playwright, but in this situation submitting the form gets you a nice URL full of parameters! Use that.
In class we always did something like item.find('h2') or the like because there was only ever one h2 we wanted. In this case there are multiple <td> tags that you want the text from! You'll need to use .find_all and treat them like normal lists ("give me the text from the first one," + "give me the text from the second one" etc etc)
It's okay to have columns you don't want or need! You can always remove them later
You'll need to grab the right <table> from the page before anything else

#### Note on try-except 

1 Instead of just if/else - tests for condition - true/false
2 In scraping, there are many ways a line can fail (missing tag, missing attribute, etc.) and you don't have to predict which — you just catch them all.
3 when return fails, just put a None in the cell and scrape the next row


#### Build 2 csvs
Include the URL in this version of the CSV. then in round 2 scraping, then use this csv's URL to build a csv for each individual cases. 


#### Ways to reach inside a css tag 

1 An HTML attribute (= written inside the tag,href, src, class) 
[] = pull a key from a dictionary

link['href'], img['src'] 

vs

2 BeautifulSoup feature/method	 
link.text,strip()              .find('a')['href']

3 
.text = "give me the words, throw away the HTML tags."

.strip() = "clean the empty space off the front and back.





## Round 1 - Failed attempt 1
<table> tag is not sufficient 

Question 1 - how to scrape table with a table - what is a good structure? 

In [24]:
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36"
}

tables = pd.read_html(
    'https://dww.des.sc.gov/DWW/JSP/SearchDispatch?number=&name=&county=All&WaterSystemType=All&SourceWaterType=All&PointOfContactType=None&SampleType=ColiformSample&begin_date=6%2F19%2F2024&end_date=6%2F19%2F2026&action=Search+For+Water+Systems',
    storage_options=headers
)


In [25]:
tables

[                                                   0  \
 0                                                NaN   
 1  Return Links  Water  System Search County Map ...   
 2                                                NaN   
 
                                                    1  
 0                              Drinking Water Branch  
 1  Water Systems  Water  System No.  Water  Syste...  
 2           Total Number of Records Displayed = 1392  ,
                    0                   1     2       3  \
 0  Water  System No.  Water  System Name  Type  Status   
 
                           4                           5  
 0  Principal  County Served  Primary  Source Water Type  ]

In [27]:
len(tables)

2

In [43]:
df = tables[1]

In [44]:
df

,0,1,2,3,4,5
0,Water System No.,Water System Name,Type,Status,Principal County Served,Primary Source Water Type


#### Other approaches to try - Try playwrite for alternative setup for form request

In [ ]:
%pip install playwright
%pip install playwright

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install playwright


Note: you may need to restart the kernel to use updated packages.


In [13]:
import sys
!{sys.executable} -m playwright install chromium

In [14]:
# Imports
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless = False)
page = await browser.new_page()

await page.goto("https://dww.des.sc.gov/DWW/")

<Response url='https://dww.des.sc.gov/DWW/' request=<Request url='https://dww.des.sc.gov/DWW/' method='GET'>>

## Solution using lxml 

In [68]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36"
}

tables = pd.read_html(
    'https://dww.des.sc.gov/DWW/JSP/SearchDispatch?number=&name=&county=All&WaterSystemType=All&SourceWaterType=All&PointOfContactType=None&SampleType=ColiformSample&begin_date=6%2F19%2F2024&end_date=6%2F19%2F2026&action=Search+For+Water+Systems',
    storage_options=headers
)

Tips: instead of pulling a table, understand the table structure here: 

In [ ]:
# <table> - Query this is not useful here
#     <tr> -- naming for headers
#     <tr> - the first case 
#         <td> case number *** use index to get to each order - [0].text.strip()
#         <td> name

In [ ]:
response = requests.get('https://dww.des.sc.gov/DWW/JSP/SearchDispatch?number=&name=&county=All&WaterSystemType=All&SourceWaterType=All&PointOfContactType=None&SampleType=ColiformSample&begin_date=6%2F19%2F2024&end_date=6%2F19%2F2026&action=Search+For+Water+Systems')
doc = BeautifulSoup(response.text, 'lxml')   # was 'html.parser'
print(response.status_code)   # 200 means OK


In [70]:
table = doc.find('table', attrs={'id': 'AutoNumber7'})

rows = []
# skip the first <tr> because it's the header, not data
for tr in table.find_all('tr')[1:]:
    cells = tr.find_all('td')      # <-- "these" are the cells

    row = {}
    row['water_system_no']   = cells[0].text.strip()
    row['water_system_name'] = cells[1].text.strip()
    row['type']              = cells[2].text.strip()
    row['status']            = cells[3].text.strip()
    row['county']            = cells[4].text.strip()
    row['source_type']       = cells[5].text.strip()

    link = cells[0].find('a')  #already at the <a> link level
    #no need to wrtie as find('a')['href']

    try:
        row['link'] = link['href'] 
    except:
        row['link'] = None


    # link = cells[0].find('a')
    # row['link'] = link['href'] if link else None
    
    rows.append(row)

len(rows)

0

In [64]:
rows

[{'water_system_no': 'SC7777',
  'water_system_name': '123 WATER',
  'type': '',
  'status': 'A',
  'county': '',
  'source_type': '',
  'link': 'WaterSystemDetail.jsp?tinwsys_is_number=7343&tinwsys_st_code=SC&wsnumber=SC7777      '},
 {'water_system_no': 'SC0874019',
  'water_system_name': '1250 OLD GILLIARD, LLC (SC0874019',
  'type': 'NC',
  'status': 'A',
  'county': 'BERKELEY',
  'source_type': 'GW',
  'link': 'WaterSystemDetail.jsp?tinwsys_is_number=7233&tinwsys_st_code=SC&wsnumber=SC0874019   '},
 {'water_system_no': 'SC2174000',
  'water_system_name': '341 EXPRESS (SC2174000)',
  'type': 'NC',
  'status': 'A',
  'county': 'FLORENCE',
  'source_type': 'GW',
  'link': 'WaterSystemDetail.jsp?tinwsys_is_number=6979&tinwsys_st_code=SC&wsnumber=SC2174000   '},
 {'water_system_no': 'SC4570950',
  'water_system_name': '4 WAY STOP (SC4570950)',
  'type': 'NC',
  'status': 'A',
  'county': 'WILLIAMSBURG',
  'source_type': 'GW',
  'link': 'WaterSystemDetail.jsp?tinwsys_is_number=6458&tinw

In [59]:
df = pd.DataFrame(rows)
df.head()

,water_system_no,water_system_name,type,status,county,source_type,link
0,SC7777,123 WATER,,A,,,None
1,SC0874019,"1250 OLD GILLIARD, LLC (SC0874019",NC,A,BERKELEY,GW,None
2,SC2174000,341 EXPRESS (SC2174000),NC,A,FLORENCE,GW,None
3,SC4570950,4 WAY STOP (SC4570950),NC,A,WILLIAMSBURG,GW,None
4,SC4060012,611 HARMON LLC (SC4060012),C,A,RICHLAND,GW,None


In [60]:
df.to_csv('water.csv', index=False)